In [1]:
# %%



#------------------------------------------------ Begin_Librairie ----------------------------------------



import pandas as pd



from bs4 import BeautifulSoup



from time import sleep



from datetime import datetime



from pandas import ExcelWriter



from selenium import webdriver



from selenium.webdriver.common.by import By



import datetime



import os



import re


import requests


import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# %%


    
    
    

In [2]:

#------------------------------------------------ Begin_ fileName ----------------------------------------



print("Running AU APRA Web Scraping Tool v.1.1")



regulatorName = 'AU APRA'



now=datetime.datetime.now()



filename= 'AU APRA SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])



scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment



os.chdir(scriptfolder)



#writer = ExcelWriter(filename)



tempfolder=os.path.join(scriptfolder, 'tempfolder') 







if os.path.exists(tempfolder):



    for rem in os.listdir(tempfolder):



        os.remove(os.path.join(tempfolder, rem))



else:



    os.mkdir(tempfolder)



Running AU APRA Web Scraping Tool v.1.1


In [3]:
# %%



#------------------------------------------------ Begin_chromedriver ----------------------------------------



#Starting Chrome driver, set to download files in tempfolder



chromeOptions = webdriver.ChromeOptions()



prefs = {"plugins.always_open_pdf_externally": True,



		 "download.prompt_for_download": False,



		 "download.default_directory" : tempfolder}



chromeOptions.add_experimental_option("prefs",prefs)



driver = webdriver.Chrome(options=chromeOptions)



driver.maximize_window()




In [ ]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


# Define a function to click an element by xpath with waiting and JS fallback

def click_element_by_xpath(driver, xpath, timeout=10):
    element = WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.XPATH, xpath)))
    # scroll into view
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", element)
    element = WebDriverWait(driver, timeout).until(EC.element_to_be_clickable((By.XPATH, xpath)))
    try:
        element.click()
    except Exception:
        driver.execute_script("arguments[0].click();", element)


def wait_for_file(folder, timeout=30):
    end_time = time.time() + timeout
    while time.time() < end_time:
        files = [f for f in os.listdir(folder)
                 if os.path.isfile(os.path.join(folder, f)) and not f.endswith('.crdownload')]
        if files:
            return files[0]
        sleep(1)
    raise TimeoutError(f"No file appeared in {folder} after {timeout}s")


def init_dict(dict_headers):
    
    dict_container = {}
    
    #dict_headers = ['Name','category','link']
    
    for header in dict_headers: dict_container[header] = {}
    
    return dict_container


def getHeaders(ths):
    
    headers = [th.text for th in ths.children if th.name == "th"]
    
    return headers




def norm(txt: str) -> str:
    return re.sub(r"\s+", " ", txt or "").strip()

def append_row(target: dict, row: dict):
    for k in target:
        target[k].append(row.get(k, ""))

def split_name_and_tech_info(li):
    raw_text = norm(li.get_text(" ", strip=True))
    p = li.find("p")
    name = norm(p.get_text(" ", strip=True)) if p else norm(raw_text.split("  ")[0])
    tech_info = norm(raw_text.replace(name, "", 1))
    return name, tech_info
# Replace your parse_li_blocks(...) with this version

def parse_tables(soup):
    
    out = []
    for table in soup.select("table"):
        accordion = table.find_parent("div", class_="js-accordion")
        p = accordion.find("p")
        h2 = accordion.find("h2")
        typology = p.get_text(strip=True) if p else (h2.get_text(strip=True) if h2 else "")

        #typology = accordion.find("p").text or accordion.find("h2").text
        headers = [norm(th.get_text(" ", strip=True)).lower() for th in table.select("tr th")]
        for tr in table.select("tr")[1:]:
            tds = [norm(td.get_text(" ", strip=True)) for td in tr.select("td")]
            if not tds:
                continue
            row = dict(zip(headers, tds))
            name = (
                row.get("name")
                or row.get("institution")
                or row.get("entity")
                or tds[0]
            )
            address = ""
            for hk, hv in row.items():
                if "address" in hk and hv:
                    address = hv
                    break
            out.append({
                "name": name.split(':')[0] if ':' in name else name,
                "category": typology,
                "address": address,
                "website": "",

            })
    return out


regCodes = ['AU_APRA_1', 'AU_APRA_2', 'AU_APRA_3', 'AU_APRA_4', 'AU_APRA_5', 'AU_APRA_6', 'AU_APRA_7','AU_APRA_9', 'AU_APRA_10', 'AU_APRA_11']



regdict = {

            'AU_APRA_1':'https://www.apra.gov.au/register-authorised-deposit-taking-institutions', 
           
            'AU_APRA_2':'https://www.apra.gov.au/register-general-insurance', 
           
            'AU_APRA_3':'https://www.apra.gov.au/register-general-insurance', 
           
            'AU_APRA_4':'https://www.apra.gov.au/register-general-insurance', 
           
            'AU_APRA_5':'https://www.apra.gov.au/register-non-operating-holding-companies', 
           
            'AU_APRA_6':'https://www.apra.gov.au/list-of-registered-life-insurers-and-friendly-societies', 
           
            'AU_APRA_7':'https://www.apra.gov.au/list-of-registered-life-insurers-and-friendly-societies', 
           
            'AU_APRA_9':'https://www.apra.gov.au/list-of-superannuation-institutions', 
           
            'AU_APRA_10':'https://www.apra.gov.au/list-of-superannuation-institutions', 
           
            'AU_APRA_11':'https://www.apra.gov.au/list-institutions-offering-retirement-savings-accounts',

             'AU_APRA_12':'https://www.apra.gov.au/list-of-registered-private-health-insurers', 
           
            'AU_APRA_13':'https://www.apra.gov.au/list-of-registered-financial-corporations'            
           
           }



Typology = {

            "AU_APRA"+"_1": "List of Authorised Deposit-taking Institutions",
            "AU_APRA"+"_2": "List of Register of general insurance",
            "AU_APRA"+"_3": "Insurers Only Authorised to Conduct Run-Off Business",
            "AU_APRA"+"_4": "Insurers that have had Insurance Authorisations Revoked (since 1st January 2002)",  
            "AU_APRA"+"_5": "Register of non-operating holding companies (NOHCs)",
            "AU_APRA"+"_6": "List of Registers of life insurance companies and friendly societies",
            "AU_APRA"+"_7": "List of Friendly Societies",
            "AU_APRA"+"_8": "Register of non-operating holding companies (NOHCs",
            "AU_APRA"+"_9": "List of RSEs",
            "AU_APRA"+"_10": "List of RSE Licensees",
            "AU_APRA"+"_11": "List of Institutions offering Retirement Savings Accounts",
            "AU_APRA"+"_12": "List of Register of private health insurers",
            "AU_APRA"+"_13": "List of Registered financial corporations",
            }

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}




zip_to_city = {
    3001: "MELBOURNE",
    2001: "SYDNEY",
    4001: "BRISBANE",
    3008: "DOCKLANDS",
    2601: "CANBERRA",
    2000: "SYDNEY",
    3000: "MELBOURNE",
    4064: "MILTON",
    2057: "CHATSWOOD",
    8007: "DARWIN",
    6904: "PERTH",
    3053: "CARLTON",
    8003: "DARWIN",
    1215: "SYDNEY",
    2300: "NEWCASTLE",
    2060: "NORTH SYDNEY",
    2190: "GREENACRE",
    3205: "SOUTH MELBOURNE",
    2002: "SYDNEY",
    2124: "NORTH RYDE",
    2500: "WOLLONGONG",
    2020: "MASCOT",
    2604: "KINGSTON",
    8001: "DARWIN",
    4000: "BRISBANE",
    3550: "BENDIGO",
    3141: "SOUTH YARRA",
    1675: "SYDNEY",
    1230: "SYDNEY"
}

processdate = now.strftime('%Y-%m-%d')


kk = 0
for k, reg in enumerate(regdict):    

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    if reg not in ['AU_APRA_9','AU_APRA_10','AU_APRA_12','AU_APRA_13']:
        # 1) Request page with verify=False
        resp = requests.get(regdict[reg], timeout=60, verify=False)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # 2) Extract entities (accordion/list first, fallback to tables)

        entities = parse_tables(soup)

        # 3) Append into sqldict
        page_title = norm(soup.title.get_text(" ", strip=True)) if soup.title else "APRA Register"

        for ent in entities:
            append_row(sqldict, {
                "Name": ent["name"],
                "Typology": ent["category"],
                "Address_1": ent["address"],
                "Website": ent["website"],
                # "Check": ent["tech_info"],            # keep technical/extra info here
                "Cntry": "AU",
                "RegCtry": "AU",
                "RegCode": "APRA",
                "RegulationType": "Regulated",
                "ListCode": reg[8:],
                "ListName": Typology[reg],
                "ListProcessDate": processdate,
            })

    elif reg == 'AU_APRA_9':
        driver.get(regdict[reg])
        sleep(5)
        driver.find_elements(By.CLASS_NAME,'document-link-container')[-1].find_element(By.TAG_NAME,'a').click()
        sleep(10)

        excel_file = wait_for_file(tempfolder)
        filePath = os.path.join(tempfolder, excel_file)
        data_9 = pd.read_excel(filePath,"List of RSE", engine='openpyxl')
        for n, abn, rn,add,tel in zip(data_9['Fund Name'], data_9['Fund ABN'], data_9['Registration Number'], data_9['Postal Address'], data_9['Contact Telephone Number']):
            sqldict['Name'].append(n)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['InternalID_1_type'].append('Fund ABN')
            sqldict['InternalID_2_type'].append('Registration Number')
            sqldict['InternalID_1'].append(abn)
            sqldict['InternalID_2'].append(rn)
            sqldict['Address_1'].append(add)
            sqldict['Phone'].append(tel)
            try:
                if int(add.split(' ')[-1]) in zip_to_city:
                    sqldict['City'].append(zip_to_city[int(add.split(' ')[-1])])
                    sqldict['Zip'].append(add.split(' ')[-1])
            except:
                sqldict['City'].append(' ')
                sqldict['Zip'].append(' ')
            sqldict['ListName'].append('List of RSEs')
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append('AU')
            sqldict['Cntry'].append('AU')
            sqldict['RegCode'].append('APRA')
            sqldict['ListCode'].append(reg[8:])
        for rem in os.listdir(tempfolder):
            os.remove(os.path.join(tempfolder, rem))

    elif reg == 'AU_APRA_10':
        driver.get(regdict[reg])
        sleep(5)
        driver.find_elements(By.CLASS_NAME,'document-link-container')[-1].find_element(By.TAG_NAME,'a').click()
        sleep(10)

        excel_file = wait_for_file(tempfolder)
        filePath = os.path.join(tempfolder, excel_file)
        data_10  = pd.read_excel(filePath,"List of Licensee", engine='openpyxl')
        for n , abn, add, ln in zip(data_10['Trustee Name'], data_10['Trustee ABN'], data_10['Postal Address'],data_10['Licence Number']):
            sqldict['Name'].append(n)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append('List of RSE Licenseess')
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append('AU')
            sqldict['Cntry'].append('AU')
            sqldict['RegCode'].append('APRA')
            sqldict['InternalID_1_type'].append('Trustee ABN')
            sqldict['InternalID_2_type'].append('Licence Number')
            sqldict['InternalID_1'].append(abn)
            sqldict['InternalID_2'].append(ln)
            sqldict['Address_1'].append(add)
            sqldict['ListCode'].append(reg[8:])
            sqldict['Phone'].append('')
            try:
                if int(add.split(' ')[-1]) in zip_to_city:
                    sqldict['City'].append(zip_to_city[int(add.split(' ')[-1])])
                    sqldict['Zip'].append(add.split(' ')[-1])
            except:
                sqldict['City'].append(' ')
                sqldict['Zip'].append(' ')
    
    elif reg == 'AU_APRA_11' :
        tables = soup.find_all("table")
        all_trs = tables[(0+kk)%2].find_all("tr")
        ths = all_trs[0]
        trs = all_trs[1:]
        headers = getHeaders(ths)
        dict_container = init_dict(headers)
        for tr in trs:
            tds = tr.find_all("td")
            for i,td in enumerate(tds):
                if i %2==0: 
                    if td.text.startswith('IMB'):
                        sqldict['Name'].append('IMB Ltd')
                    else:
                        if td.text[-1] == '1':
                            sqldict['Name'].append(td.text[:len(td.text)-1])
                        else:
                            sqldict['Name'].append(td.text)
                    
                    sqldict['ListName'].append('List of Institutions offering Retirement Savings Accounts')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append('AU')
                    sqldict['Cntry'].append('AU')
                    sqldict['RegCode'].append('APRA')
                    sqldict['ListCode'].append(reg[8:])
                    sqldict = bourange_same_length_array(sqldict)    
        kk += 1
    
    elif reg == 'AU_APRA_13':
        resp = requests.get(regdict[reg], timeout=60, verify=False)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        tables = soup.find_all("table")
        for table in tables:
            tbody = table.find('tbody')
            for tr in tbody.find_all('tr'):
                tds = tr.find_all('td')
                name = tds[0].text
                abn_code = tds[1].text
                sqldict['Name'].append(name)
                sqldict['InternalID_1'].append(abn_code)
                sqldict['InternalID_1_type'].append('ABN')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append('AU')
                sqldict['Cntry'].append('AU')
                sqldict['RegCode'].append('APRA')
                sqldict['ListCode'].append(reg[8:])
                sqldict['ListName'].append('List of Registered financial corporations')
    elif reg == 'AU_APRA_12':
        resp = requests.get(regdict[reg], timeout=60, verify=False)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        cards = soup.select("div.js-accordion.accordion")
        print("AU_APRA_12 cards found:", len(cards))  # debug

        for card in cards:

            address = ""
            phone = ""
            email_ = ""
            web_ = ""
            name = ""

            for tr in card.select("div.accordion__content table tr"):
                th = tr.find("th")
                td = tr.find("td")
                # print(th.get_text(" ", strip=True), td.get_text(" ", strip=True))
                key = th.get_text(" ", strip=True).lower()
                val = td.get_text(" ", strip=True)
                # print("Key:", key)  # debug
                if 'name' in key:
                    name = val
                    sqldict['Name'].append(name)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append('AU')
                    sqldict['Cntry'].append('AU')
                    sqldict['RegCode'].append('APRA')
                    sqldict['ListCode'].append('12')
                    sqldict['ListName'].append('List of Register of private health insurers')
                elif "address" in key:
                    address = val
                    sqldict['Address_1'].append(address)
                elif "telephone" in key:
                    phone = val
                    sqldict['Phone'].append(phone)
                elif "email" in key:
                    email_ = val
                    sqldict['Email'].append(email_)
                elif "web site" in key or "website" in key:
                    a = td.find("a", href=True)
                    web_ = a["href"].strip() if a else val
                    sqldict['Website'].append(web_)
                #print(f"Extracted - Name: {name}, Address: {address}, Phone: {phone}, Email: {email_}, Website: {web_}")  # debug
                
                
        

 # debug
        sqldict = bourange_same_length_array(sqldict)
    
                        
sqldict = bourange_same_length_array(sqldict)        
        
        
    

[INFO] : Working 1/12 _(AU_APRA_1)_ 
[INFO] : Working 2/12 _(AU_APRA_2)_ 
[INFO] : Working 3/12 _(AU_APRA_3)_ 
[INFO] : Working 4/12 _(AU_APRA_4)_ 
[INFO] : Working 5/12 _(AU_APRA_5)_ 
[INFO] : Working 6/12 _(AU_APRA_6)_ 
[INFO] : Working 7/12 _(AU_APRA_7)_ 
[INFO] : Working 8/12 _(AU_APRA_9)_ 
[INFO] : Working 9/12 _(AU_APRA_10)_ 
[INFO] : Working 10/12 _(AU_APRA_11)_ 
[INFO] : Working 11/12 _(AU_APRA_12)_ 
AU_APRA_12 cards found: 28
Key: membership type
Key: registered name
Key: restriction
Key: address
Key: states in which insurer operates
Key: telephone
Key: email
Key: web site
Key: membership type
Key: registered name
Key: address
Key: states in which insurer operates
Key: telephone
Key: web site
Key: membership type
Key: registered name
Key: address
Key: states in which insurer operates
Key: telephone
Key: web site
Key: membership type
Key: registered name
Key: address
Key: states in which insurer operates
Key: telephone
Key: web site
Key: membership type
Key: registered name
Key

In [ ]:

# %%



#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------



os.chdir(scriptfolder)

df = pd.DataFrame(sqldict)   # or your parsed list

# make sure Typology text is clean first
df["Typology"] = df["Typology"].astype(str).str.strip()
df = df.drop_duplicates(subset=["Name", "Typology"], keep="first").reset_index(drop=True)

rules = {
    "Authorised to conduct new or renewal insurance business": {
        "ListCode": "2",
        "ListName": "Authorised to conduct new or renewal insurance business",
        "RegulationType": "Regulated",
    },
    "Insurers only authorised to conduct run-off business": {
        "ListCode": "3",
        "ListName": "Insurers only authorised to conduct run-off business",
        "RegulationType": "Regulated",
    },
    "Revoked insurance authorisations (since 1 January 2002)": {
        "ListCode": "4",
        "ListName": "Insurers that have had Insurance Authorisations Revoked (since 1st January 2002)",
        "RegulationType": "Revoked",
    },
    "Insurers deregistered under the Corporations Act 2001": {
        "ListCode": "4",
        "ListName": "Insurers deregistered under the Corporations Act 2001",
        "RegulationType": "Deregisted",  # keep as requested
    },
    "Life insurance companies": {
        "ListCode": "6",
        "ListName": "List of Registers of life insurance companies",
        "RegulationType": "Regulated",
    },
    "Friendly societies": {
        "ListCode": "7",
        "ListName": "List of Friendly Societies",
        "RegulationType": "Regulated",
    },
}

df["ListCode"] = df["Typology"].map({k: v["ListCode"] for k, v in rules.items()}).fillna(df.get("ListCode", ""))
df["ListName"] = df["Typology"].map({k: v["ListName"] for k, v in rules.items()}).fillna(df.get("ListName", ""))
df["RegulationType"] = df["Typology"].map({k: v["RegulationType"] for k, v in rules.items()}).fillna(df.get("RegulationType", ""))

df = df.drop_duplicates(subset=["Name", "Typology"], keep="first").reset_index(drop=True)

df.to_excel(filename, index=False)
driver.quit()


sleep(3)

   